In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
root_dir = '/content'
os.chdir(root_dir)

In [4]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep='\t',
    names=['Label','Message']
)

In [5]:
df.head()

,Label,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Label    5572 non-null   object
 1   Message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [7]:
encoder = LabelEncoder()

df['Label'] = encoder.fit_transform(df['Label'])

In [8]:
df.head()

,Label,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df['Message'])

y = df['Label']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

In [12]:
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))

print(confusion_matrix(y_test, nb_pred))

print(classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.9856502242152466
[[957   9]
 [  7 142]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.94      0.95      0.95       149

    accuracy                           0.99      1115
   macro avg       0.97      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [13]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [14]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

print(confusion_matrix(y_test, lr_pred))

print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.9883408071748879
[[966   0]
 [ 13 136]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       1.00      0.91      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [15]:
nb_score = cross_val_score(nb, X, y, cv=5)

print("Naive Bayes CV Accuracy:", nb_score.mean())

Naive Bayes CV Accuracy: 0.979899364790558


In [16]:
lr_score = cross_val_score(lr, X, y, cv=5)

print("Logistic Regression CV Accuracy:", lr_score.mean())

Logistic Regression CV Accuracy: 0.9813348254180386


In [17]:
params = {
    'alpha':[0.1,0.5,1.0,2.0]
}

grid_nb = GridSearchCV(
    MultinomialNB(),
    params,
    cv=5
)

grid_nb.fit(X_train,y_train)

print("Best Parameters:",grid_nb.best_params_)

print("Best Score:",grid_nb.best_score_)

Best Parameters: {'alpha': 1.0}
Best Score: 0.9804789298062839


In [18]:
params = {
    'C':[0.01,0.1,1,10],
    'solver':['liblinear']
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000),
    params,
    cv=5
)

grid_lr.fit(X_train,y_train)

print("Best Parameters:",grid_lr.best_params_)

print("Best Score:",grid_lr.best_score_)

Best Parameters: {'C': 10, 'solver': 'liblinear'}
Best Score: 0.981376042437328


In [19]:
print("Naive Bayes Accuracy :", accuracy_score(y_test, nb_pred))
print("Logistic Regression Accuracy :", accuracy_score(y_test, lr_pred))

print("Naive Bayes CV :", nb_score.mean())
print("Logistic Regression CV :", lr_score.mean())

print("Best NB Score :", grid_nb.best_score_)
print("Best LR Score :", grid_lr.best_score_)

Naive Bayes Accuracy : 0.9856502242152466
Logistic Regression Accuracy : 0.9883408071748879
Naive Bayes CV : 0.979899364790558
Logistic Regression CV : 0.9813348254180386
Best NB Score : 0.9804789298062839
Best LR Score : 0.981376042437328
